# 3.2.1 Price Dynamics and Volatility (Light)

This notebook is strictly **load-only**. All heavy simulations and precomputations must be generated beforehand and stored under `data/processed/remote_results/`.

By default, the notebook compares the real market with `QRU`, `QR`, and `FTQR`. `SAQR` is optional and is loaded only when `USE_SAQR = True`.


## 1. Setup

The notebook uses only compact remote artifacts. It never calibrates a model, never simulates a path, and never rebuilds a cache locally.

Required files:

- `price_dynamics_metadata.json`
- `plot_window_series.parquet`
- `volatility_summary.csv`
- `volatility_summary_aggregated.csv`


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "data/processed/remote_results"
META_FILE = RESULTS_DIR / "price_dynamics_metadata.json"
PLOT_FILE = RESULTS_DIR / "plot_window_series.parquet"
VOL_FILE = RESULTS_DIR / "volatility_summary.csv"
VOL_AGG_FILE = RESULTS_DIR / "volatility_summary_aggregated.csv"

USE_SAQR = False
BASE_MODEL_ORDER = ["Real", "QRU", "QR", "FTQR"]

MODEL_COLORS = {
    "Real": "#111111",
    "QRU": "#4c78a8",
    "QR": "#f58518",
    "FTQR": "#54a24b",
    "SAQR": "#b279a2",
}
PERIOD_ORDER = ["Full day", "Calm", "Active"]

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing precomputed file {path}. Run the remote precompute script on gpu.enst.fr."
        )
    return path

metadata = json.loads(require_file(META_FILE).read_text())
plot_df = pd.read_parquet(require_file(PLOT_FILE))
plot_df["timestamp"] = pd.to_datetime(plot_df["timestamp"])
vol_df = pd.read_csv(require_file(VOL_FILE))
vol_agg_df = pd.read_csv(require_file(VOL_AGG_FILE))

available_models = ["Real"] + [m for m in metadata.get("models", []) if m != "Real"]
model_order = BASE_MODEL_ORDER.copy()
if USE_SAQR:
    if "SAQR" not in available_models:
        raise FileNotFoundError(
            "Missing SAQR precomputed artifacts in data/processed/remote_results. "
            "Rebuild remote results with --models QRU QR FTQR SAQR if SAQR is required."
        )
    model_order.append("SAQR")

missing_models = [m for m in model_order if m not in available_models]
if missing_models:
    raise FileNotFoundError(
        "Missing precomputed model outputs for: " + ", ".join(missing_models) +
        ". Run the remote precompute script on gpu.enst.fr."
    )

plot_df = plot_df[plot_df["model"].isin(model_order)].copy()
vol_df = vol_df[vol_df["model"].isin([m for m in model_order if m != "Real"])].copy()
vol_agg_df = vol_agg_df[vol_agg_df["model"].isin([m for m in model_order if m != "Real"])].copy()

summary_df = pd.DataFrame({
    "Plot day": [metadata["plot_day"]],
    "Plot window": [f"{metadata['plot_start']} to {metadata['plot_end']}"] ,
    "Benchmark days": [len(metadata["stats_days"])],
    "Loaded models": [", ".join(model_order)],
    "USE_SAQR": [USE_SAQR],
})
display(summary_df)


## 2. One-Day Visual Comparison

The representative-day window is precomputed remotely. The top panel shows the mid-price path and the bottom panel reports the corresponding 10-minute rolling annualized volatility.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for model in model_order:
    sub = plot_df[plot_df["model"] == model].copy()
    axes[0].plot(sub["timestamp"], sub["mid_price"], label=model, color=MODEL_COLORS[model], linewidth=1.8)
axes[0].set_title(f"Mid-Price Dynamics on {metadata['plot_day']} ({metadata['plot_start']}–{metadata['plot_end']})")
axes[0].set_ylabel("Mid-price")
axes[0].grid(alpha=0.2)
axes[0].legend(ncol=len(model_order), frameon=False)

for model in model_order:
    sub = plot_df[plot_df["model"] == model].copy()
    axes[1].plot(sub["timestamp"], sub["rolling_vol"], label=model, color=MODEL_COLORS[model], linewidth=1.8)
axes[1].set_title("Rolling Annualized Volatility (10-minute window, 1-second sampling)")
axes[1].set_ylabel("Annualized volatility")
axes[1].set_xlabel("Time")
axes[1].grid(alpha=0.2)
fig.tight_layout()

calm_window = (
    vol_df[(vol_df["day"] == metadata["plot_day"]) & (vol_df["period"] == "Calm")]
    [["model", "sigma_real", "sigma_sim", "relative_difference_pct", "quadratic_error"]]
    .sort_values("model")
)
display(calm_window.style.format({
    "sigma_real": "{:.4f}",
    "sigma_sim": "{:.4f}",
    "relative_difference_pct": "{:+.2f}",
    "quadratic_error": "{:.6f}",
}))


## 3. Multi-Day Volatility Benchmark

The table below is entirely precomputed from remote one-second mid-price series. The local notebook only formats and displays the results.


In [ ]:
vol_agg_df["period"] = pd.Categorical(vol_agg_df["period"], categories=PERIOD_ORDER, ordered=True)
vol_agg_df["model"] = pd.Categorical(vol_agg_df["model"], categories=[m for m in model_order if m != "Real"], ordered=True)
vol_agg_df = vol_agg_df.sort_values(["period", "model"]).reset_index(drop=True)

display(vol_agg_df.style.format({
    "avg_relative_difference_pct": "{:+.2f}",
    "avg_quadratic_error": "{:.6f}",
    "mean_sigma_real": "{:.4f}",
    "mean_sigma_sim": "{:.4f}",
}))

best_by_period = vol_agg_df.loc[vol_agg_df.groupby("period")["avg_quadratic_error"].idxmin(), ["period", "model"]]
best_lines = "\n".join([f"- **{row.period}**: {row.model}" for row in best_by_period.itertuples(index=False)])
display(Markdown(f"""
### Interpretation

- The ranking is computed on identical precomputed windows for all loaded models.
- `SAQR` is excluded by default to keep the notebook instantly usable even when SAQR artifacts have not been generated yet.
- Best model by average quadratic error in each period:
{best_lines}
"""))


## 4. Summary

This light notebook is designed for safe local use. It requires only small precomputed files and remains fully usable without SAQR unless SAQR is explicitly requested.
